In [ ]:
!pip install torch torchvision

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights
from torch.utils.data import DataLoader
from google.colab import drive

In [ ]:
# Redimensiona a imagem para 224x244, o tamanho padrão esperado pelo ResNet-18
# Em seguida converte a imagem para tensor, passando os valores para o intervalo entre 0 e 1
# E por fim normaliza os dados com media 0.5 e desvio padrão 0.5 alterando os valores para o intervalo entre -1 e +1
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
])

In [ ]:
drive.mount('/content/drive')

# Os dados estão compartilhados através do link https://drive.google.com/drive/folders/1HSjQ5INhzDQIQAlt7C644uj7lvwD3mHa?usp=sharing
# Faça download dos dados para seu google drive e ajuste o caminho com o caminho da pasta do dataset
dataset_dir = "/content/drive/MyDrive/PosIA/dataset/Breast_Cancer_Biopsy_7500"


Mounted at /content/drive


In [ ]:
# Carregar dataset com ImageFolder
# O ImageFolder interpreta cada subpasta como uma classe (no caso as classes benign e malignant)
full_dataset = datasets.ImageFolder(root=dataset_dir, transform=transform)

# Divisão 80% treino / 20% teste
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [train_size, test_size])

# recebe amostras de 32 por vez embaralhando a cada época
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
#Se se cuda.is_available usa GPU, caso contrário usa cpu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Utiliza o Resnet com 18 pesos
# Isto significa que será utilizada 1 camada convolucional inicial + 16 camadas convolucionais  + 1 camada final totalmente conectada
modelo = models.resnet18(weights=ResNet18_Weights.DEFAULT)

# Recupera a quanidade de features geradas pelas camadas anteriores e usadas na entrada da camada fully conected
num_features = modelo.fc.in_features

# Altera a camada fully conected para usar uma camada linear que recebe a quantidade de features geradas pelas camadas anteriores como entrada e gera sáida 2 saídas (benigno/maligno)
modelo.fc = nn.Linear(num_features, 2)

# garante que todos os parâmetros e tensores internos do modelo fiquem na GPU (se disponível) ou na CPU.
modelo = modelo.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 165MB/s]


In [ ]:
# função de perda para classificação; compara a saída do modelo com a classe correta e gera um valor do erro que guia o treinamento
criterio = nn.CrossEntropyLoss()

# usa o optimizador Adam para fazer ajustes nos parâmetros conforme o erro calculado por CrossEntropyLoss
otimizador = optim.Adam(modelo.parameters(), lr=0.001)

# repete o processo de treinamento 5 vezes, passando por todo o conjunto de dados em cada repetição
for epoch in range(5):
    # prepara o modelo para treino
    modelo.train()
    # inicializa a variável para acumular a quantidade de erros e acertos
    perda = 0.0
    corretos = 0
    total = 0

    # looping no dataset de treino
    for images, labels in train_loader:
        # envia a imagem e o label para o device utilizado, garantindo que o modelo e os dados utilizarão o mesmo dispositivo (GPU ou CPU)
        images, labels = images.to(device), labels.to(device)

        # zera o gradiente do optimizador Adam a cada conjunto de 32 imagens processadas
        otimizador.zero_grad()

        # passa as imagens pelo modelo e recupera a predição
        prediction = modelo(images)

        # calcula a perda comparando a predição com o rótulo correto (label)
        loss = criterio(prediction, labels)

        # calcula o gradiente de erro dos parâmetros
        loss.backward()

        # aplica o gradiente e atualiza os parâmetros
        otimizador.step()

        # acumula o erro
        perda += loss.item()

        # recupera a classe com maior score para cada linha da predição
        _, predicted_classes = torch.max(prediction, 1)
        # compara os dois array (preditos e os rotúlos corretos) e soma os valores iguais
        corretos += (predicted_classes == labels).sum().item()
        # acumula a quantidade de acertos
        total += labels.size(0)

    # calcula média da perda e acurácia
    media_erro = perda / len(train_loader)
    acuracia = corretos / total

    # Exibe o erro e a acurácia para cada processo de treinamento (epoch)
    print(f"Epoch {epoch+1}, Erro: {media_erro:.4f}, Acurácia treino: {acuracia:.2%}")


Epoch 1, Erro: 0.2845, Acurácia treino: 88.53%
Epoch 2, Erro: 0.1534, Acurácia treino: 94.12%
Epoch 3, Erro: 0.1212, Acurácia treino: 95.62%
Epoch 4, Erro: 0.1107, Acurácia treino: 96.02%
Epoch 5, Erro: 0.0716, Acurácia treino: 97.37%


In [ ]:
# prepara o modelo para teste
modelo.eval()
# inicializa a variável para acumular a quantidade de acertos
corretos = 0
total = 0

# Indica que não usará gradientes pois não está em treinamento
# Isto torna o processamento mais leve
with torch.no_grad():
    # looping no dataset de teste
    for images, labels in test_loader:
        # envia a imagem e o label para o device utilizado, garantindo que o modelo e os dados utilizarão o mesmo dispositivo (GPU ou CPU)
        images, labels = images.to(device), labels.to(device)
        # passa as imagens pelo modelo e recupera a predição
        prediction = modelo(images)
        # recupera a classe com maior score para cada linha da predição
        _, predicted = torch.max(prediction.data, 1)
        # compara os dois array (preditos e os rotúlos corretos) e soma os valores iguais
        corretos += (predicted == labels).sum().item()
        # acumula a quantidade de acertos
        total += labels.size(0)

print("Acurácia no conjunto de teste:", 100 * corretos / total)

Acurácia no conjunto de teste: 86.94203864090606
